In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yummyooo123/www2025-mmctr-data")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'www2025-mmctr-data' dataset.
Path to dataset files: /kaggle/input/www2025-mmctr-data


In [ ]:
!pip install polars transformers torch pillow scikit-learn

In [ ]:
import polars as pl
import torch
import numpy as np
from transformers import AutoProcessor, AutoModel
from PIL import Image
import os
from pathlib import Path
from sklearn.decomposition import PCA
import warnings
from torch.utils.data import Dataset, DataLoader
from concurrent.futures import ThreadPoolExecutor
import time
warnings.filterwarnings('ignore')

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
torch.set_grad_enabled(False)  # Disable gradients for inference

# 1. Load data
print("Loading data...")
item_features = pl.read_parquet("/kaggle/input/www2025-mmctr-data/MicroLens_1M_MMCTR/item_feature.parquet")
item_info = pl.read_parquet("/kaggle/input/www2025-mmctr-data/MicroLens_1M_MMCTR/MicroLens_1M_x1/item_info.parquet")
print(f"Loaded {len(item_features)} items")

# 2. Load model once
print("Loading SigLIP model...")
model = AutoModel.from_pretrained("google/siglip-so400m-patch14-384").to(device)
processor = AutoProcessor.from_pretrained("google/siglip-so400m-patch14-384")
model.eval()

# 3. Create a dataset class for efficient batching
class ItemDataset(Dataset):
    def __init__(self, df, images_dir):
        self.df = df
        self.images_dir = Path(images_dir)
        self.image_paths = []

        # Precompute image paths
        for item_id in df['item_id'].to_list():
            img_path = self.images_dir / f"{item_id}.jpg"
            self.image_paths.append(str(img_path) if img_path.exists() else None)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.row(idx, named=True)
        text = f"{row['item_title']} {' '.join(map(str, row['item_tags']))}".strip()
        img_path = self.image_paths[idx]
        return row['item_id'], text, img_path

# 4. Batch processing function with image preloading
def preload_images_batch(image_paths, max_workers=4):
    """Preload images in parallel"""
    def load_image(path):
        if path and os.path.exists(path):
            try:
                return Image.open(path).convert("RGB")
            except:
                return None
        return None

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        images = list(executor.map(load_image, image_paths))
    return images

def process_batch(batch_data, device):
    """Process a batch of items"""
    item_ids, texts, image_paths = zip(*batch_data)

    # Process texts in batch
    text_inputs = processor(
        text=texts,
        padding="max_length",
        max_length=64,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        text_features = model.get_text_features(**text_inputs)

    # Find valid images
    valid_indices = [i for i, path in enumerate(image_paths) if path and os.path.exists(path)]

    if valid_indices:
        # Preload valid images in parallel
        valid_paths = [image_paths[i] for i in valid_indices]
        valid_images = preload_images_batch(valid_paths, max_workers=8)

        # Filter out None images
        valid_indices = [valid_indices[i] for i, img in enumerate(valid_images) if img is not None]
        valid_images = [img for img in valid_images if img is not None]

        if valid_images:
            # Process images in batch
            image_inputs = processor(
                images=valid_images,
                return_tensors="pt"
            ).to(device)

            with torch.no_grad():
                batch_image_features = model.get_image_features(**image_inputs)

            # Combine features
            combined_features = torch.zeros_like(text_features)

            for i, idx in enumerate(valid_indices):
                combined_features[idx] = 0.6 * text_features[idx] + 0.4 * batch_image_features[i]

            # For items without images: text only
            no_image_indices = [i for i in range(len(batch_data)) if i not in valid_indices]
            for idx in no_image_indices:
                combined_features[idx] = text_features[idx]

            return item_ids, combined_features.cpu().numpy()

    # If no valid images, return text features only
    return item_ids, text_features.cpu().numpy()

print("\nProcessing items with optimized batching...")
images_dir = "/kaggle/input/www2025-mmctr-data/MicroLens_1M_MMCTR/item_images/item_images"

# Create dataset
dataset = ItemDataset(item_features, images_dir)

batch_size = 128  # Increased from 32
all_embeddings = []
processed_ids = []
total_batches = (len(dataset) + batch_size - 1) // batch_size

start_time = time.time()

for batch_idx in range(0, len(dataset), batch_size):
    batch_end = min(batch_idx + batch_size, len(dataset))
    batch_data = [dataset[i] for i in range(batch_idx, batch_end)]

    print(f"Processing batch {batch_idx//batch_size + 1}/{total_batches}", end='\r')

    item_ids, batch_embeddings = process_batch(batch_data, device)

    processed_ids.extend(item_ids)
    all_embeddings.append(batch_embeddings)

# Concatenate all embeddings
all_embeddings = np.vstack(all_embeddings)
print(f"\nExtracted embeddings shape: {all_embeddings.shape}")
print(f"Time taken: {time.time() - start_time:.2f} seconds")

# 6. Dimensionality reduction (optimized)
print("\nReducing dimensions...")
start_time = time.time()

from sklearn.decomposition import IncrementalPCA

if len(all_embeddings) > 50000:  # Use IncrementalPCA for large datasets
    pca = IncrementalPCA(n_components=128, batch_size=1024)
    # Fit in batches
    for i in range(0, len(all_embeddings), 1024):
        batch = all_embeddings[i:i+1024]
        pca.partial_fit(batch)
    reduced_embeddings = pca.transform(all_embeddings)
else:
    pca = PCA(n_components=128, random_state=42)
    reduced_embeddings = pca.fit_transform(all_embeddings)

print(f"Reduced embeddings shape: {reduced_embeddings.shape}")
print(f"Explained variance: {pca.explained_variance_ratio_.sum():.4f}")
print(f"PCA time: {time.time() - start_time:.2f} seconds")

# 7. Update item_info (optimized lookup)
print("\nUpdating item_info.parquet...")

embeddings_dict = dict(zip(processed_ids, reduced_embeddings))

# Use vectorized lookup instead of loop
item_ids_array = item_info['item_id'].to_numpy()
new_embeddings = np.zeros((len(item_info), 128), dtype=np.float32)

# Create a mask for items with embeddings
mask = np.isin(item_ids_array, processed_ids)
matching_ids = item_ids_array[mask]

id_to_idx = {id_: idx for idx, id_ in enumerate(processed_ids)}
matching_indices = [id_to_idx.get(id_, -1) for id_ in matching_ids]
valid_mask = np.array([idx != -1 for idx in matching_indices])

if np.any(valid_mask):
    new_embeddings[mask][valid_mask] = reduced_embeddings[np.array(matching_indices)[valid_mask]]

embeddings_df = pl.DataFrame(new_embeddings, schema=[f"emb_{i}" for i in range(128)])
item_info = item_info.with_columns(embeddings_df)

if "item_emb_d128" in item_info.columns:
    item_info = item_info.drop("item_emb_d128")

item_info = item_info.with_columns(
    pl.concat_list([pl.col(f"emb_{i}") for i in range(128)]).alias("item_emb_d128")
).drop([f"emb_{i}" for i in range(128)])

print("Saving files...")
item_info.write_parquet("/kaggle/working/Updated_iteminfo.parquet")

# Save embeddings separately
embeddings_save = pl.DataFrame({
    "item_id": processed_ids,
    "item_emb_d128": list(reduced_embeddings)
})
embeddings_save.write_parquet("/kaggle/working/item_embeddings_siglip_128.parquet")


print(f"Processed {len(processed_ids)} items")
print(f"Total time: {time.time() - start_time:.2f} seconds")

Using device: cuda
Loading data...
Loaded 91717 items
Loading SigLIP model...

Processing items with optimized batching...
